In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import tempfile
from urllib.request import urlretrieve

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')

# SQLAlchemy 2.1+ за замовчуванням обирає драйвер psycopg (v3) для 'postgresql://'.
# Явно вказуємо psycopg2, щоб notebook працював однаково в будь-якій версії SQLAlchemy.
uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(uri, future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
def run(sql: str):
    """Виконує SQL-скрипт (кілька statement через ';') в одній транзакції.
    Якщо останній statement повертає рядки — повертає їх як DataFrame."""
    statements = [s.strip() for s in sql.split(';') if s.strip()]
    result = None
    with engine.begin() as conn:
        for stmt in statements:
            res = conn.execute(text(stmt))
            if res.returns_rows:
                result = pd.DataFrame(res.fetchall(), columns=list(res.keys()))
    return result


def explain(sql: str):
    """Виконує EXPLAIN (ANALYZE, BUFFERS) і друкує план як текст."""
    with engine.connect() as conn:
        rows = conn.execute(text(sql)).fetchall()
    print('\n'.join(r[0] for r in rows))

In [4]:
DATA_DIR = '/content' if os.path.isdir('/content') else tempfile.gettempdir()
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/'
MONTHS = ['2024-02', '2024-03']
SAMPLE_PER_MONTH = 300_000

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

try:
    frames = []
    for month in MONTHS:
        fname = f'yellow_tripdata_{month}.parquet'
        path = os.path.join(DATA_DIR, fname)
        if not os.path.exists(path):
            urlretrieve(BASE_URL + fname, path)
        part = pd.read_parquet(path, columns=columns)
        frames.append(part.sample(n=min(SAMPLE_PER_MONTH, len(part)), random_state=42))
    df = pd.concat(frames, ignore_index=True)

    # Basic cleaning before loading to PostgreSQL
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
    df = df[df['PULocationID'].notna()]

    df.to_sql('nyc_taxi_raw', engine, if_exists='replace', index=False, chunksize=10_000)
    DATA_SOURCE = 'nyc_tlc'
except Exception as exc:
    print(f'NYC TLC download/load failed: {type(exc).__name__}: {exc}')
    print('-> using stub-schema fallback (Variant B)')
    DATA_SOURCE = 'stub'

print('DATA_SOURCE =', DATA_SOURCE)

NYC TLC download/load failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>
-> using stub-schema fallback (Variant B)
DATA_SOURCE = stub


In [5]:
if DATA_SOURCE == 'nyc_tlc':
    display(run('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw'))

In [6]:
SQL_DAILY_TLC = """
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
  AND tpep_pickup_datetime >= TIMESTAMP '2024-02-01'
  AND tpep_pickup_datetime <  TIMESTAMP '2024-04-01'
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day)
"""

SQL_DAILY_STUB = """
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id  INTEGER NOT NULL,
    ts_day          DATE    NOT NULL,
    trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
    total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day)
);

INSERT INTO t7_hw_taxi_daily (
    pickup_zone_id, ts_day, trip_count, avg_fare, total_distance
)
SELECT
    z.zone_id,
    DATE '2024-03-31' - (59 - n) AS ts_day,
    500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,
    (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
    (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance
FROM generate_series(0, 59) AS n
CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id)
"""

run(SQL_DAILY_TLC if DATA_SOURCE == 'nyc_tlc' else SQL_DAILY_STUB)

run("""
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10
""")

,pickup_zone_id,n_days
0,1,60
1,2,60
2,3,60
3,4,60
4,5,60


In [7]:
run('''
SELECT COUNT(*)                       AS n_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day)                    AS min_day,
       MAX(ts_day)                    AS max_day
FROM t7_hw_taxi_daily
''')

,n_rows,n_zones,min_day,max_day
0,300,5,2024-02-01,2024-03-31


In [8]:
if DATA_SOURCE == 'nyc_tlc':
    display(run("""
    DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        (tpep_pickup_datetime::TIMESTAMP  AT TIME ZONE 'America/New_York') AS pickup_at,
        (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10, 2) AS trip_distance,
        fare_amount::NUMERIC(10, 2)   AS fare_amount,
        passenger_count::INTEGER      AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100;

    SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 5
    """))
else:
    print('Stub-schema: сирих поїздок немає, typed-таблицю пропускаємо.')

Stub-schema: сирих поїздок немає, typed-таблицю пропускаємо.


In [9]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-02-01,550,NaN,NaN,None
1,1,2024-02-02,587,550.0,NaN,None
2,1,2024-02-03,624,587.0,NaN,None
3,1,2024-02-04,661,624.0,NaN,None
4,1,2024-02-05,698,661.0,NaN,None
5,1,2024-02-06,585,698.0,NaN,None
6,1,2024-02-07,622,585.0,NaN,None
7,1,2024-02-08,659,622.0,550.0,None
8,1,2024-02-09,696,659.0,587.0,None
9,1,2024-02-10,733,696.0,624.0,None


In [10]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    ROUND(AVG(avg_fare) OVER w7, 4) AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-02-01,13.20,13.2000,1
1,1,2024-02-02,13.60,13.4000,2
2,1,2024-02-03,14.00,13.6000,3
3,1,2024-02-04,14.40,13.8000,4
4,1,2024-02-05,13.20,13.6800,5
5,1,2024-02-06,13.60,13.6667,6
6,1,2024-02-07,14.00,13.7143,7
7,1,2024-02-08,14.40,13.8857,7
8,1,2024-02-09,13.20,13.8286,7
9,1,2024-02-10,13.60,13.7714,7


In [11]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    ROUND(STDDEV_SAMP(avg_fare) OVER w30, 4) AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-02-01,13.20,None,1
1,1,2024-02-02,13.60,0.2828,2
2,1,2024-02-03,14.00,0.4000,3
3,1,2024-02-04,14.40,0.5164,4
4,1,2024-02-05,13.20,0.5215,5
5,1,2024-02-06,13.60,0.4676,6
6,1,2024-02-07,14.00,0.4451,7
7,1,2024-02-08,14.40,0.4781,8
8,1,2024-02-09,13.20,0.4899,9
9,1,2024-02-10,13.60,0.4638,10


In [12]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    ROUND(AVG(trip_count) OVER we, 2) AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-02-01,550,550,550.00
1,1,2024-02-02,587,1137,568.50
2,1,2024-02-03,624,1761,587.00
3,1,2024-02-04,661,2422,605.50
4,1,2024-02-05,698,3120,624.00
5,1,2024-02-06,585,3705,617.50
6,1,2024-02-07,622,4327,618.14
7,1,2024-02-08,659,4986,623.25
8,1,2024-02-09,696,5682,631.33
9,1,2024-02-10,733,6415,641.50


In [13]:
run("""
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-02,760,14.00,2370.00,873,801,550,13.7142857142857143,0.44256306778301919370,22105,713.0645161290322581
1,1,2024-03-03,797,14.40,2395.00,760,838,587,13.8857142857142857,0.45404946816424076179,22902,715.6875000000000000
2,1,2024-03-04,834,13.20,2420.00,797,725,624,13.8285714285714286,0.46683084140056408733,23736,719.2727272727272727
3,1,2024-03-05,871,13.60,2445.00,834,762,661,13.7714285714285714,0.45404946816424076179,24607,723.7352941176470588
4,1,2024-03-06,908,14.00,2470.00,871,799,698,13.7142857142857143,0.44256306778301919370,25515,729.0000000000000000
5,1,2024-03-07,795,14.40,2495.00,908,836,585,13.8857142857142857,0.45404946816424076179,26310,730.8333333333333333
6,1,2024-03-08,832,13.20,2520.00,795,873,622,13.8285714285714286,0.46683084140056408733,27142,733.5675675675675676
7,1,2024-03-09,869,13.60,2545.00,832,760,659,13.7714285714285714,0.45404946816424076179,28011,737.1315789473684211
8,1,2024-03-10,906,14.00,2570.00,869,797,696,13.7142857142857143,0.44256306778301919370,28917,741.4615384615384615
9,1,2024-03-11,943,14.40,2595.00,906,834,733,13.8857142857142857,0.45404946816424076179,29860,746.5000000000000000


In [14]:
run("""
SELECT COUNT(*) AS n_training_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day) AS first_day_with_full_history
FROM v_t7_hw_taxi_features
""")

,n_training_rows,n_zones,first_day_with_full_history
0,150,5,2024-03-02


In [15]:
run("""
WITH lagged AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS lag_1d_correct,
        LAG(trip_count, 1) OVER (ORDER BY pickup_zone_id, ts_day)            AS lag_1d_leaky,
        ROW_NUMBER()       OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS rn
    FROM t7_hw_taxi_daily
)
SELECT pickup_zone_id, ts_day, trip_count, lag_1d_correct, lag_1d_leaky
FROM lagged
WHERE rn = 1
ORDER BY pickup_zone_id
LIMIT 5
""")

,pickup_zone_id,ts_day,trip_count,lag_1d_correct,lag_1d_leaky
0,1,2024-02-01,550,None,NaN
1,2,2024-02-01,600,None,1083.0
2,3,2024-02-01,650,None,1133.0
3,4,2024-02-01,700,None,1183.0
4,5,2024-02-01,750,None,1233.0


In [16]:
run("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type
""")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [17]:
# Різна структура payload для кожного типу події
run("""
SELECT DISTINCT ON (event_type) event_type, payload
FROM t7_hw_events
ORDER BY event_type, event_id
""")

,event_type,payload
0,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
2,click,"{'device': 'desktop', 'page_url': '/category/2', 'element_id': 'btn_2'}"
3,page_view,"{'url': '/product/6', 'device': 'tablet', 'referrer': 'direct'}"
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"


In [18]:
run("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10
""")

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 03:00:00+02:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
1,18,19,2024-03-14 03:00:00+02:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
2,33,34,2024-02-28 03:00:00+02:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
3,48,49,2024-02-13 03:00:00+02:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
4,63,64,2024-03-29 03:00:00+02:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
5,78,79,2024-03-14 03:00:00+02:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
6,93,94,2024-02-28 03:00:00+02:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
7,108,109,2024-02-13 03:00:00+02:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
8,123,124,2024-03-29 03:00:00+02:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
9,138,139,2024-03-14 03:00:00+02:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


In [19]:
run("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10
""")

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [20]:
run("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10
""")

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


In [21]:
run("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15
""")

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [22]:
run("""
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10
""")

,event_id,order_id,total
0,253,o_100253,303.00
1,258,o_100258,308.00
2,263,o_100263,313.00
3,268,o_100268,318.00
4,273,o_100273,323.00
5,278,o_100278,328.00
6,283,o_100283,333.00
7,288,o_100288,338.00
8,293,o_100293,343.00
9,298,o_100298,348.00


In [23]:
run("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
ANALYZE t7_hw_events
""")

explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1759.06..1759.07 rows=1 width=8) (actual time=39.940..39.942 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3624 width=0) (actual time=0.013..39.731 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.164 ms
Execution Time: 39.966 ms


In [24]:
run("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events'))                       AS table_size
""")

,gin_index_size,table_size
0,1144 kB,8000 kB


In [25]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1072.61..1072.62 rows=1 width=8) (actual time=4.749..4.750 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=26.62..1066.04 rows=2628 width=0) (actual time=0.457..4.575 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..25.96 rows=2628 width=0) (actual time=0.341..0.342 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=27
Planning Time: 0.233 ms
Execution Time: 4.777 ms


In [26]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google')
""")

Aggregate  (cost=1061.11..1061.12 rows=1 width=8) (actual time=4.315..4.317 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=23.17..1056.18 rows=1971 width=0) (actual time=0.361..4.182 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..22.68 rows=1971 width=0) (actual time=0.250..0.251 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.133 ms
Execution Time: 4.354 ms


In [27]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=16.829..16.831 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.023..15.984 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.089 ms
Execution Time: 16.936 ms


In [28]:
run("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_device_btree')) AS btree_index_size
""")

,btree_index_size
0,360 kB


In [29]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1242.24..1242.25 rows=1 width=8) (actual time=3.631..3.633 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.73..1221.46 rows=8315 width=0) (actual time=0.449..3.227 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.65 rows=8315 width=0) (actual time=0.333..0.333 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=18 read=1
Planning Time: 0.238 ms
Execution Time: 3.664 ms


In [30]:
import time

INSERT_BATCH = """
INSERT INTO {tbl} (user_id, event_ts, event_type, payload)
SELECT user_id, event_ts, event_type, payload
FROM t7_hw_events
  -- усі 50 000 подій
"""

timings = {}
for label, with_gin in [('no_index', False), ('gin_jsonb_path_ops', True)]:
    run("""
    DROP TABLE IF EXISTS t7_hw_events_write_test;
    CREATE TABLE t7_hw_events_write_test (LIKE t7_hw_events INCLUDING DEFAULTS INCLUDING IDENTITY)
    """)
    if with_gin:
        run("CREATE INDEX ON t7_hw_events_write_test USING GIN (payload jsonb_path_ops)")
    t0 = time.perf_counter()
    run(INSERT_BATCH.format(tbl='t7_hw_events_write_test'))
    timings[label] = round((time.perf_counter() - t0) * 1000, 1)

run("DROP TABLE IF EXISTS t7_hw_events_write_test")
pd.DataFrame([timings], index=['insert 50k rows, ms'])

,no_index,gin_jsonb_path_ops
"insert 50k rows, ms",119.0,263.4


In [31]:
run("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts, order_id, total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm
""")

,n_checkouts
0,10000


In [32]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=1959.32..1959.33 rows=3 width=72) (actual time=11.768..11.771 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1959.32..1973.38 rows=5624 width=72) (actual time=11.766..11.767 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1802.28..1886.63 rows=5624 width=72) (actual time=11.693..11.756 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.33 rows=10130 width=132) (actual time=0.011..7.545 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=14 read=2
Planning Time: 0.151 ms
Execution Time: 11.842

In [33]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=3.685..3.687 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=3.683..3.685 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=3.673..3.675 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.007..1.001 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=34
Planning Time: 0.174 ms
Execution Time: 3.717 ms


In [34]:
# Перевірка: обидві версії дають однаковий результат
jsonb_res = run("""
SELECT payload ->> 'payment_method' AS payment_method,
       ROUND(AVG((payload ->> 'total')::NUMERIC), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_events WHERE event_type = 'checkout'
GROUP BY 1 ORDER BY avg_total DESC LIMIT 3
""")
norm_res = run("""
SELECT payment_method, ROUND(AVG(total), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY 1 ORDER BY avg_total DESC LIMIT 3
""")
display(jsonb_res)
print('Results identical:', jsonb_res.equals(norm_res))

,payment_method,avg_total,n
0,paypal,300.52,3333
1,card,300.50,3334
2,crypto,300.48,3333


Results identical: True


In [35]:
run("""
SELECT 'jsonb: t7_hw_events (all 5 types)' AS storage,
       pg_size_pretty(pg_relation_size('t7_hw_events')) AS heap_size
UNION ALL
SELECT 'normalized: t7_hw_checkouts_norm',
       pg_size_pretty(pg_relation_size('t7_hw_checkouts_norm'))
""")

,storage,heap_size
0,jsonb: t7_hw_events (all 5 types),8000 kB
1,normalized: t7_hw_checkouts_norm,752 kB


## Reflection

*Цифри нижче взято з output-ів клітинок цього notebook (PostgreSQL 16.2, 50 000 подій у `t7_hw_events`). Після повторного запуску час трохи відрізнятиметься, але оператори плану мають залишитися тими самими.*

**1. GIN: до і після.** Без індексу `payload @> {"payment_method":"card"}` виконувався через **Seq Scan**: cost `0.00..1750.00`, прочитано всі 1000 сторінок, *Rows Removed by Filter: 46 666*, Execution Time ≈ **40 ms**. Після `GIN (jsonb_path_ops)` (розмір 1144 kB при heap 8000 kB) план змінився на **Bitmap Index Scan -> Bitmap Heap Scan**: total cost впав до `1072.62`, сам індекс прочитав лише 4 буфери, Execution Time ≈ **4.8 ms** (у ~8 разів швидше). Запит `{"referrer":"google"}` теж пішов через Bitmap Index Scan (≈ 4.4 ms, 2500 рядків). Heap Blocks = 1000, бо потрібні рядки розкидані по всіх сторінках таблиці, і виграш дає саме відмова від перевірки 50 000 JSON-документів.

**2. Чому GIN не підходить для `payload ->> 'device' = 'mobile'`.** GIN індексує структуру документа і працює лише з операторами свого opclass (`@>`, `@?`, `@@`). `->>` + `=` - це порівняння *результату виразу* з текстом, тому planner обрав **Seq Scan** (≈ 16.9 ms) і помилився в оцінці: `rows=250` замість фактичних 8333, бо для виразу немає статистики. Expression B-tree `((payload ->> 'device'))` (360 kB) дав Bitmap Index Scan, ≈ **3.7 ms**, а після `ANALYZE` оцінка стала точною (`rows=8315`).

**3. Read/write trade-off.** GIN прискорює containment- і JSONPath-читання, але кожен INSERT розкладає документ на багато ключів індексу. У моєму вимірі вставка 50 000 подій зайняла **119 ms без індексу та 263 ms з GIN** (≈ 2.2×). До цього додаються +1.1 MB диска, а також VACUUM і pending list.

**4. JSONB чи нормалізована таблиця.** Для однакового агрегату по checkout JSONB-версія читала 1000 буферів (Seq Scan по всіх подіях, *Rows Removed: 40 000*, ≈ 11.8 ms). Normalized-версія читала **94 буфери** (752 kB), ≈ **3.7 ms**, результати збіглися. JSONB я обрав би для сирого event-log з типами, що часто змінюються, а нормалізовану таблицю - для curated ML-features і стабільних аналітичних запитів, де потрібні типи, `CHECK` та точна статистика.

**5. Новий event-type `refund`.** У JSONB-підході міграція не потрібна: достатньо одного `INSERT` з `event_type='refund'` і payload `{order_id, amount, reason}`; GIN індексує нові ключі автоматично. У normalized-підході потрібні `CREATE TABLE t7_hw_refunds_norm` (FK на `order_id`, `CHECK amount > 0`), індекси, backfill з events, зміна ETL та версіонування схеми.

**6. Time-series features.** Демо в Завданні 2 показало, навіщо потрібен `PARTITION BY pickup_zone_id`. Без нього перший день зони 2 отримав `lag_1d_leaky = 1083`, тобто останнє значення зони 1, замість NULL. Це тихий leakage історії іншої сутності. Після фільтра `trips_lag_30d IS NOT NULL` у training-view залишилося 150 рядків (5 зон × 30 днів, з 2024-03-02). На real TLC `LAG(30)` рахує рядки, а не календарні дні, тому для розріджених зон потрібен gap-fill.


## Checklist

- [x] pgserver запущено, показано версію PostgreSQL
- [x] `t7_hw_taxi_daily` (> 100 рядків), daily aggregation на рівні `pickup_zone_id, ts_day`
- [x] lag 1d / 7d / 30d, rolling 7d avg (`RANGE … INTERVAL '6 days'`), rolling 30d std, expanding sum/avg
- [x] Усі windows містять `PARTITION BY pickup_zone_id`
- [x] `t7_hw_events` з 5 event-типами та різними JSONB-схемами
- [x] Запити `@>`, `?` / `?|`, `jsonb_path_query` / `jsonb_path_exists`
- [x] `EXPLAIN (ANALYZE, BUFFERS)` до/після GIN і до/після expression B-tree
- [x] Normalized checkout-таблиця + аналітичний запит у двох версіях
- [x] Bonus MongoDB pseudocode у markdown
- [x] Reflection